# India's Primary-Care AMR Surveillance Gap: Quantification & Analysis

**Prepared for:** Lipi (AI-native primary-care/OPD service, India)
**Purpose:** Quantify the gap between where antibiotics are consumed in India (primary care/community) and
where AMR surveillance specimens are actually sourced (hospital/tertiary care), using real, cited public data.

Every number in this notebook is tagged inline to its exact source (report, PMID/DOI, page/table where available).
Where public data does not support a precise national figure, that is stated explicitly as a GAP FLAG rather than
estimated or assumed.

**Data provenance:** Source extraction was performed by three research passes that fetched and parsed primary
documents (WHO GLASS API/reports, ICMR AMRSN PDFs via icmr.gov.in, NCDC NARS-Net PDFs via ncdc.mohfw.gov.in, and
PubMed/PMC full texts for consumption studies). Raw structured extractions are saved as companion JSON artifacts:
`who_glass_india.json`, `icmr_ncdc_india.json`, `consumption_india.json`.

## 1. Load source extractions

Each JSON file is a structured, per-field extraction with an inline `citation` object (source title, URL,
page/table). These were produced by dedicated research passes against primary documents (not from memory).

In [1]:
import json, host
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

who_glass_path = host.artifact_path("df96d730-8e91-44cf-b85a-551b3ece4b2f")
icmr_ncdc_path = host.artifact_path("d2fa05c4-fd46-4132-bae5-e68c7bf5758f")
consumption_path = host.artifact_path("61e24e3b-b473-4c32-8e74-8446d1e3b07c")

with open(who_glass_path) as f:
    who_glass = json.load(f)
with open(icmr_ncdc_path) as f:
    icmr_ncdc = json.load(f)
with open(consumption_path) as f:
    consumption = json.load(f)

print("WHO GLASS records:", len(who_glass["records"]))
print("ICMR/NCDC records:", len(icmr_ncdc["records"]), "| state sites:", len(icmr_ncdc["state_level_site_distribution"]))
print("Consumption records:", len(consumption["records"]), "| state data:", len(consumption["state_level_data"]))

WHO GLASS records: 34
ICMR/NCDC records: 25 | state sites: 31
Consumption records: 10 | state data: 10


## 2. WHO GLASS — India-specific data availability (2016/2018/2020/2022 report years)

**Key finding: India-specific sentinel-site counts, site-type breakdowns, and specimen-type breakdowns are
NOT publicly available** via the WHO Global Health Observatory (GHO) OData API for any of the four requested
report years — the GHO's GLASS site-count indicators (`AMRGLASS_SURVL01-05`) expose only **region-level**
aggregates (e.g. South-East Asia Region), not country rows for India. Full country-level detail would require
manual extraction from each report's PDF country-profile annex, which was not completed within the time budget
of this pass — flagged explicitly rather than estimated.

Source: `who_glass_india.json`, `narrative_notes` field (GHO API queried directly: `ghoapi.azureedge.net/api/AMRGLASS_SURVL01..05`).

**What IS available:** India-level time series for two (of the four requested) pathogen-antibiotic pairs, via
GHO indicators `AMR_INFECT_ECOLI` (E. coli resistance to **3rd-generation cephalosporins** — not fluoroquinolones,
which is not a country-disaggregated GHO indicator) and `AMR_INFECT_MRSA` (S. aureus methicillin resistance).

In [2]:
glass_amr = [r for r in who_glass["records"] if isinstance(r["value"], (int, float))]
glass_df = pd.DataFrame(glass_amr)[["report_year", "data_year", "field", "value"]]
glass_df

   report_year                                                             data_year                                                         field   value
0         2018  2017 (INFERRED, not independently confirmed for this report edition)  ecoli_3rd_gen_cephalosporin_resistance_india_ADJACENT_METRIC   75.11
1         2018  2017 (INFERRED, not independently confirmed for this report edition)                                            mrsa_percent_india   52.50
2         2020  2018 (INFERRED, not independently confirmed for this report edition)  ecoli_3rd_gen_cephalosporin_resistance_india_ADJACENT_METRIC   78.83
3         2020  2018 (INFERRED, not independently confirmed for this report edition)                                            mrsa_percent_india   63.10
4         2022                                                                  2020  ecoli_3rd_gen_cephalosporin_resistance_india_ADJACENT_METRIC   86.81
5         2022                                                        

> **Gap flags (explicit, from `who_glass_india.json`):** E. coli fluoroquinolone resistance, K. pneumoniae
> carbapenem resistance, and S. pneumoniae penicillin resistance are **NOT PUBLICLY REPORTED** for India at
> country level in the WHO GHO API for any of the 4 requested years. Sentinel-site counts and site-type
> breakdowns for India are likewise **NOT PUBLICLY REPORTED** via this channel.

## 3. ICMR AMRSN + NCDC NARS-Net — India's flagship AMR surveillance networks

Extracted by direct PDF download and text-mining of `icmr.gov.in` and `ncdc.mohfw.gov.in` primary documents
(source: `icmr_ncdc_india.json`).

In [3]:
icmr_years = {2019: 29, 2020: 28, 2022: 25, 2023: 21}   # ICMR AMRSN sentinel hospitals per report year
nars_years = {2017: 13, 2018: 16, 2019: 21, 2020: 29, 2021: 35, 2022: 36, 2023: 41, 2024: 54}  # NCDC NARS-Net sites

surveillance_summary = []
for yr, n in icmr_years.items():
    surveillance_summary.append({"network": "ICMR AMRSN", "report_year": yr, "total_sentinel_sites": n, "primary_care_sites": 0})
for yr, n in nars_years.items():
    surveillance_summary.append({"network": "NCDC NARS-Net", "report_year": yr, "total_sentinel_sites": n, "primary_care_sites": 0})

surv_df = pd.DataFrame(surveillance_summary)
surv_df

          network  report_year  total_sentinel_sites  primary_care_sites
0      ICMR AMRSN         2019                    29                   0
1      ICMR AMRSN         2020                    28                   0
2      ICMR AMRSN         2022                    25                   0
3      ICMR AMRSN         2023                    21                   0
4   NCDC NARS-Net         2017                    13                   0
5   NCDC NARS-Net         2018                    16                   0
6   NCDC NARS-Net         2019                    21                   0
7   NCDC NARS-Net         2020                    29                   0
8   NCDC NARS-Net         2021                    35                   0
9   NCDC NARS-Net         2022                    36                   0
10  NCDC NARS-Net         2023                    41                   0
11  NCDC NARS-Net         2024                    54                   0

**Source citations (verbatim quotes from primary PDFs, `icmr_ncdc_india.json`):**

- ICMR AMRSN 2020/2022/2023 annual reports: *"the data presented in this report is not reflective of the
  community levels of AMR in the country"* — [icmr.gov.in AMRSN_annual_report_2020_1.pdf](https://www.icmr.gov.in/icmrobject/custom_data/pdf/resource-guidelines/AMRSN_annual_report_2020_1.pdf), p.4-5;
  [AMRSN_Annual_Report_2022.pdf](https://www.icmr.gov.in/icmrobject/custom_data/pdf/resource-guidelines/AMRSN_Annual_Report_2022.pdf), p.4;
  [1725536060_annual_report_2023.pdf](https://www.icmr.gov.in/icmrobject/uploads/Documents/1725536060_annual_report_2023.pdf), p.4.
- ICMR AMRSN 2023: Annexure I lists exactly **4 Nodal Centers + 17 Regional Centers = 21 sentinel hospitals**, all
  medical-college/tertiary hospitals (AIIMS New Delhi, CMC Vellore, JIPMER Puducherry, PGIMER Chandigarh + 17 named
  regional centers) — p.241, Annexure I.
- NCDC NARS-Net 2023: *"generates annual reports on National AMR Surveillance data from tertiary care facilities
  since 2018"* — [ncdc.mohfw.gov.in/uploads/pdf/amr32.pdf](https://ncdc.mohfw.gov.in/uploads/pdf/amr32.pdf), Section 1.1.
  Every Annexure I site list examined (2017-2024) contains exclusively government medical colleges, AIIMS/PGIMER/JIPMER-type
  institutes, and a handful of large private tertiary hospitals — **zero PHC, CHC, district-hospital, or
  outpatient-only sites** in any year.

**Conclusion: `surveillance_from_primary_care_pct = 0%`** — not an estimate, but an exact count confirmed across
12 site-years of primary-source ICMR AMRSN and NCDC NARS-Net reports (2017–2024).

In [4]:
surveillance_from_primary_care_pct = 0.0
print(f"surveillance_from_primary_care_pct = {surveillance_from_primary_care_pct}%")
print("Basis: 0 of 21-29 ICMR AMRSN sentinel hospitals (2019-2023) and 0 of 13-54 NCDC NARS-Net sites")
print("(2017-2024) are primary-care/community facilities, in every year examined.")

surveillance_from_primary_care_pct = 0.0%
Basis: 0 of 21-29 ICMR AMRSN sentinel hospitals (2019-2023) and 0 of 13-54 NCDC NARS-Net sites
(2017-2024) are primary-care/community facilities, in every year examined.


## 4. Antibiotic consumption — primary care / community share

**Key finding: no single source gives a clean, nationally representative hospital-vs-community percentage
split for India.** India's largest consumption datasets (PharmaTrac) are private-sector retail-pharmacy sales
audits that cannot distinguish community from hospital-administered use at the point of care (source:
`consumption_india.json`, `narrative_notes`). We therefore triangulate across four independent, non-equivalent
proxy estimates and report a **range**, with the most-cited direct claim as the best point estimate.

In [5]:
consumption_estimates = [
    {"estimate_pct": 80, "label": "Community share of antibiotic use (direct, most-cited claim)",
     "source": "Kotwani & Holloway, BMC Infect Dis 2011 (PMC3097160)",
     "quote": "About 80% of antibiotics are used in the community and the rest are used in hospitals"},
    {"estimate_pct": 85, "label": "PharmaTrac retail-pharmacy sales-channel share (distribution-channel proxy)",
     "source": "Koya et al., Lancet Reg Health SEA 2022 (PMID 37383993)",
     "quote": "85% retail pharmacy / 15% hospitals & dispensing doctors (sales-channel, not point-of-care)"},
    {"estimate_pct": 65, "label": "Single rural district hospital (Anantapur, AP): outpatient share of DDDs",
     "source": "PMC4100948",
     "quote": "Outpatient prescriptions accounted for 2/3 of the overall antibiotic consumption"},
    {"estimate_pct": 87.5, "label": "Private-sector share of ALL drug prescriptions (not antibiotic-specific proxy)",
     "source": "National Health Accounts 2015-16, cited in Koya et al. 2022",
     "quote": "85-90% of all drug prescriptions happen in the private sector"},
]
for e in consumption_estimates:
    print(f"{e['estimate_pct']:>5}%  {e['label']}")
    print(f"        [{e['source']}] \"{e['quote']}\"")

pcts = [e["estimate_pct"] for e in consumption_estimates]
consumption_min, consumption_max = min(pcts), max(pcts)
consumption_best_estimate = 80.0   # Kotwani & Holloway 2011 -- most direct, most-cited claim in Indian OPD literature

print(f"\nconsumption_in_primary_care_pct: range=[{consumption_min}, {consumption_max}]%, best_estimate={consumption_best_estimate}%")

   80%  Community share of antibiotic use (direct, most-cited claim)
        [Kotwani & Holloway, BMC Infect Dis 2011 (PMC3097160)] "About 80% of antibiotics are used in the community and the rest are used in hospitals"
   85%  PharmaTrac retail-pharmacy sales-channel share (distribution-channel proxy)
        [Koya et al., Lancet Reg Health SEA 2022 (PMID 37383993)] "85% retail pharmacy / 15% hospitals & dispensing doctors (sales-channel, not point-of-care)"
   65%  Single rural district hospital (Anantapur, AP): outpatient share of DDDs
        [PMC4100948] "Outpatient prescriptions accounted for 2/3 of the overall antibiotic consumption"
 87.5%  Private-sector share of ALL drug prescriptions (not antibiotic-specific proxy)
        [National Health Accounts 2015-16, cited in Koya et al. 2022] "85-90% of all drug prescriptions happen in the private sector"

consumption_in_primary_care_pct: range=[65, 87.5]%, best_estimate=80.0%


> **Caveat:** the 80% figure traces to citations [1,2] in Kotwani & Holloway (2011) rather than a primary
> India-specific national audit; it is corroborated in magnitude (not methodology) by the PharmaTrac 85%
> retail-channel proxy. Treat as directionally reliable, not a precise national statistic — this is stated
> explicitly rather than presented as a settled number.

## 5. THE GAP NUMBER

$$\text{GAP} = \text{consumption\_in\_primary\_care\_pct} - \text{surveillance\_from\_primary\_care\_pct}$$

In [6]:
surveillance_pct_mean = 0.0     # exact, confirmed across 12 site-years (Section 3)
consumption_pct_best = consumption_best_estimate   # 80.0 (Section 4)

GAP_NUMBER = consumption_pct_best - surveillance_pct_mean
GAP_MIN = consumption_min - surveillance_pct_mean
GAP_MAX = consumption_max - surveillance_pct_mean

print("="*66)
print("THE GAP NUMBER")
print("="*66)
print(f"consumption_in_primary_care_pct (best estimate) = {consumption_pct_best}%")
print(f"surveillance_from_primary_care_pct              = {surveillance_pct_mean}%")
print(f"GAP NUMBER = {GAP_NUMBER} percentage points")
print(f"Plausible range (driven by consumption-side estimate spread) = [{GAP_MIN}, {GAP_MAX}] pp")

THE GAP NUMBER
consumption_in_primary_care_pct (best estimate) = 80.0%
surveillance_from_primary_care_pct              = 0.0%
GAP NUMBER = 80.0 percentage points
Plausible range (driven by consumption-side estimate spread) = [65.0, 87.5] pp


> **On the "95% range" request:** the four consumption estimates are heterogeneous point estimates from
> different methodologies (patient exit interviews, sales-channel proxy, single-facility chart review, national
> health accounts) — not repeated measurements of the same quantity with sampling error. A frequentist 95% CI is
> not statistically defensible from n=4 non-commensurable estimates. We report the **min–max plausible range
> across independent methodologies** (65–87.5 pp) as the honest uncertainty proxy instead, since the
> surveillance-side term is an exact, zero-uncertainty count.

### Headline: **THE GAP = 80 percentage points** (plausible range 65–87.5 pp)

Roughly 80% of India's antibiotic use happens in primary care/community settings, while **0%** of specimens
feeding India's two flagship AMR surveillance networks (ICMR AMRSN, NCDC NARS-Net) come from primary-care sites.

## 6. Chart 1 — Where antibiotics are consumed vs. where AMR surveillance happens

In [7]:
focal_c, comp_c = "#C0392B", "#a98261"
fig, ax = plt.subplots(figsize=(7.2, 5.0))
categories = ["Where antibiotics\nare consumed", "Where AMR surveillance\nspecimens come from"]
community_vals = [consumption_pct_best, surveillance_pct_mean]
hospital_vals = [100 - consumption_pct_best, 100 - surveillance_pct_mean]
x = np.arange(len(categories))
width = 0.55
ax.bar(x, community_vals, width, label="Community / primary care", color=focal_c)
ax.bar(x, hospital_vals, width, bottom=community_vals, label="Hospital / tertiary care", color=comp_c)
for i, (c, h) in enumerate(zip(community_vals, hospital_vals)):
    ax.text(x[i], c/2, f"{c:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold",
            color="white" if c > 12 else "black")
    ax.text(x[i], c + h/2, f"{h:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold",
            color="white" if h > 12 else "black")
ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylabel("Share (%)"); ax.set_ylim(0, 108)
ax.set_title("India's antibiotic use is community-driven; its AMR surveillance is not", fontsize=11)
ax.errorbar([x[0]], [consumption_pct_best],
            yerr=[[consumption_pct_best-consumption_min],[consumption_max-consumption_pct_best]],
            fmt="none", ecolor="black", elinewidth=1.2, capsize=5, zorder=5)
ax.text(x[0]+0.32, consumption_pct_best, f"range across\nsources: {consumption_min:.0f}-{consumption_max:.0f}%",
        fontsize=6.5, va="center", ha="left", color="#333333")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=8)
fig.text(0.5, 0.01,
         "Consumption: Kotwani & Holloway 2011 (PMC3097160); range across PharmaTrac proxy (Koya et al. 2022) and\n"
         "single-facility studies. Surveillance: ICMR AMRSN + NCDC NARS-Net extraction (icmr_ncdc_india.json).",
         ha="center", va="bottom", fontsize=6, color="#555555")
fig.subplots_adjust(bottom=0.28)
fig.savefig("chart1_consumption_vs_surveillance.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Chart 2 — AMRSN / NARS-Net sentinel site time series

In [8]:
c_nars, c_icmr = "#2E6F95", "#a98261"
fig, ax = plt.subplots(figsize=(8.0, 5.0))
nars_yrs = sorted(nars_years.keys()); nars_counts = [nars_years[y] for y in nars_yrs]
icmr_yrs = sorted(icmr_years.keys()); icmr_counts = [icmr_years[y] for y in icmr_yrs]
ax.plot(nars_yrs, nars_counts, marker="o", color=c_nars, linewidth=2, zorder=3)
ax.plot(icmr_yrs, icmr_counts, marker="s", color=c_icmr, linewidth=2, zorder=3)
ax.text(nars_yrs[-1]+0.15, nars_counts[-1], "NCDC NARS-Net", color=c_nars, fontsize=9, va="center", fontweight="bold")
ax.text(icmr_yrs[-1]+0.15, icmr_counts[-1], "ICMR AMRSN", color=c_icmr, fontsize=9, va="center", fontweight="bold")
ax.annotate("Every site added in both networks, every year (2017-2024),\nis a tertiary/medical-college hospital "
            "-- zero expansion\ntoward primary care or community sites",
            xy=(2021, 35), xytext=(2017.3, 48), fontsize=7.5, color="#333333",
            arrowprops=dict(arrowstyle="-", color="#888888", lw=0.8))
ax.set_xlabel("Report year"); ax.set_ylabel("Number of sentinel sites")
ax.set_xlim(2016.5, 2025.5); ax.set_ylim(0, 60); ax.set_xticks(list(range(2017, 2025)))
ax.set_title("AMR sentinel networks have grown 3-4x since 2017 -- with zero primary-care sites added", fontsize=10.5)
fig.text(0.5, 0.01,
         "Source: icmr_ncdc_india.json extraction from ICMR AMRSN (icmr.gov.in) and NCDC NARS-Net\n"
         "(ncdc.mohfw.gov.in, amr30-39.pdf) annual reports, 2017-2024.",
         ha="center", va="bottom", fontsize=6, color="#555555")
fig.subplots_adjust(bottom=0.20, right=0.88)
fig.savefig("chart2_sentinel_site_timeseries.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Chart 3 — State-level geographic blind spot

In [9]:
import matplotlib.lines as mlines
consumption_state_did = {"Delhi": 23.5, "Punjab": 22.9, "Telangana": 15.3, "Odisha": 8.9,
                          "Jharkhand": 8.5, "Rajasthan": 8.3, "Bihar": 8.1, "Madhya Pradesh": 7.2}
nars_state_sites_2023 = {d["state"]: d["num_sites"] for d in icmr_ncdc["state_level_site_distribution"]}
states = list(consumption_state_did.keys())
x_consumption = [consumption_state_did[s] for s in states]
y_sites = [nars_state_sites_2023.get(s, 0) for s in states]
high_focus = {"Bihar", "Jharkhand", "Madhya Pradesh", "Odisha", "Rajasthan"}
colors = ["#C0392B" if s in high_focus else "#2E6F95" for s in states]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(x_consumption, y_sites, s=110, c=colors, edgecolor="white", linewidth=0.8, zorder=3)
label_offsets = {"Delhi": (8,4), "Punjab": (8,4), "Telangana": (8,4), "Rajasthan": (10,2),
                 "Madhya Pradesh": (10,-2), "Bihar": (-8,-14), "Jharkhand": (6,10), "Odisha": (8,-16)}
for s, xv, yv in zip(states, x_consumption, y_sites):
    dx, dy = label_offsets[s]
    ax.annotate(s, (xv, yv), textcoords="offset points", xytext=(dx, dy), fontsize=8,
                ha="left" if dx >= 0 else "right")
ax.set_xlabel("Private-sector antibiotic consumption, median 2011-2019 (DID = DDD/1,000 pop/day)")
ax.set_ylabel("NCDC NARS-Net sentinel sites, 2023")
ax.set_title("Geographic blind spot: state antibiotic use vs. AMR surveillance coverage", fontsize=10.5)
ax.set_xlim(5, 27); ax.set_ylim(-0.7, 4.2)
ax.axvline(15, color="#999999", linestyle=":", linewidth=0.8, zorder=1)
ax.text(15.3, 4.0, "High-Focus states (red) -- lower absolute\nconsumption but WORSENING antibiotic-quality\n"
        "trend (JAC-AMR 2022) and 1-3 sentinel sites each", fontsize=7, color="#333333", va="top")
leg1 = mlines.Line2D([], [], color="#C0392B", marker="o", linestyle="None", markersize=8, label="High-Focus state (NHM classification)")
leg2 = mlines.Line2D([], [], color="#2E6F95", marker="o", linestyle="None", markersize=8, label="Non-High-Focus state")
ax.legend(handles=[leg1, leg2], frameon=False, loc="lower right", fontsize=7.5)
fig.text(0.5, 0.01,
         "Consumption: JAC-AMR 2022 (PMID 36320447), PharmaTrac private-sector retail sales, 2011-2019 medians.\n"
         "Surveillance: NCDC NARS-Net 2023 annual report, Annexure I site list.",
         ha="center", va="bottom", fontsize=6, color="#555555")
fig.subplots_adjust(bottom=0.22)
fig.savefig("chart3_state_blindspot_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Summary & product implication

- **THE GAP NUMBER = 80 percentage points** (plausible range 65–87.5 pp): the difference between the share of
  India's antibiotic consumption happening in primary care (~80%, best estimate) and the share of AMR surveillance
  specimens sourced from primary care (0%, exact, confirmed 2017–2024).
- Both of India's flagship AMR networks (ICMR AMRSN, NCDC NARS-Net) are **exclusively tertiary-care/medical-college
  hospital networks**, by their own explicit statement, in every report examined.
- WHO GLASS India-specific site/specimen granularity is **not publicly accessible** at the country level via the
  GHO API — a distinct, separate transparency gap from the ICMR/NCDC finding.
- **Product implication:** no existing national AMR surveillance network captures outpatient/primary-care
  antibiogram or resistance-pattern data for India. Lipi's doctor-confirmed, longitudinal OPD prescribing data is
  a potential differentiated signal for local antimicrobial resistance trends — but this is a **hypothesis to be
  validated** (see `amr_study_protocol.md`), not an established capability, since Lipi's data streams are
  prescriptions/diagnoses, not microbiology culture results.